In [ ]:
    ############    #############   AsyncIO and async-await   #############   ##############   

 =>  Phase: P0 -- FDE Foundations & Engineering Baseline
 =>  Topic: 0.1 Production Python
 =>  Week:  Week 1
 =>  Track: Core

 =>  Sections to fill in:
       1. Theory
       2. Diagram(s) / flowchart(s) (images/ folder)  -- only where the concept needs one
       3. Command / config reference (if applicable)
       4. Runnable code demo(s)
       5. Hands-on lab checklist
       6. Common pitfalls / notes


In [ ]:
    ############    #############   AsyncIO and async-await   #############   ##############   

 =>  async def defines a coroutine function; calling it returns a coroutine object, it does
       not run the body immediately.

 =>  await suspends the current coroutine and hands control back to the event loop until the
       awaited thing (another coroutine, a socket read, a sleep) is ready.

 =>  The event loop is a single-threaded scheduler. It runs whichever coroutine is ready and
       switches away the moment one awaits something that isn't ready yet.

 =>  asyncio.gather(*coros) runs coroutines concurrently and waits for all of them.

      Syntax -->

                async def fetch(name, delay):
                      await asyncio.sleep(delay)
                      return name

                await asyncio.gather(fetch('a', 1), fetch('b', 1))


In [ ]:
import asyncio
import time

async def fetch(name: str, delay: float) -> str:
    print(f"[{name}] starting, will take {delay}s")
    await asyncio.sleep(delay)
    print(f"[{name}] done")
    return name

async def main():
    start = time.perf_counter()
    results = await asyncio.gather(fetch("a", 1), fetch("b", 1), fetch("c", 1))
    elapsed = time.perf_counter() - start
    print("results:", results)
    print(f"elapsed: {elapsed:.2f}s (would be ~3s if run sequentially)")

await main()  # in a .py script use: asyncio.run(main())


<img src="images/asyncio-event-loop.png" alt="Sequential vs concurrent execution timeline">

In [ ]:
 =>  All three fetch() calls start almost immediately, then sleep concurrently, so total
       elapsed time is ~1s instead of ~3s.

 =>  In a real service, 'await asyncio.sleep()' is standing in for any I/O wait: a DB query,
       an HTTP call to an LLM provider, a file read.

 =>  Rule of thumb: async helps when you are waiting on I/O, not when you are burning CPU.
       CPU-bound work still blocks the single event-loop thread unless offloaded (see the
       Concurrency vs parallelism notebook).


In [ ]:
    ############    #############   Hands-on Lab Checklist   #############   ##############   

 =>  [ ] Replace asyncio.sleep with a real async HTTP call (httpx.AsyncClient) to a public
           API and time 3 concurrent calls vs 3 sequential ones.

 =>  [ ] Build a tiny FastAPI endpoint that awaits 2 slow calls with asyncio.gather instead
           of one after another.


In [ ]:
    ############    #############   Common Pitfalls   #############   ##############   

 =>  Calling a blocking function (requests.get, time.sleep, a sync DB driver) inside an
       async def -- this blocks the ENTIRE event loop, freezing every other coroutine too.

 =>  Forgetting to await a coroutine -- Python just creates a coroutine object and warns
       'coroutine was never awaited', the code inside never runs.

 =>  Assuming asyncio.gather magically uses multiple CPU cores -- it doesn't; it's
       concurrency on one thread, not parallelism (next notebook).
